# Setup
## Colab Setup
Only use when running the code via Colab!

In [ ]:
# clone the repo and load packages
!git clone https://github.com/dambedn/llm_refusal_explained.git 2>/dev/null || (cd llm_refusal_explained && git pull)
%cd llm_refusal_explained

In [ ]:
!pip list --format=freeze | grep -v -i -E "^(transformers|accelerate|datasets)==" > /content/constraints.txt
!pip install -q -r requirements-colab.txt -c /content/constraints.txt

## Local Setup
Only use when running the code locally.
Please take care to use the cooresponding .venv kernel after the uv packages have been installed.

In [3]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

from datasets import load_dataset

ModuleNotFoundError: No module named 'matplotlib'

# Load the OR-Bench Dataset, a Gemma Model

## Load the OR-Bench Dataset

In [ ]:
# Load model: copied from the respective Hugging Face page: 
#https://huggingface.co/google/gemma-3-1b-it?library=transformers
# First login with hugging face token
!hf auth login

In [ ]:
# Load dataset: copied from the respective Hugging Face page
from datasets import load_dataset

ds = load_dataset("bench-llm/or-bench", "or-bench-80k")

In [ ]:
# Example input
ds["train"][0]

In [ ]:
ds_hard = load_dataset("bench-llm/or-bench", "or-bench-hard-1k")

In [ ]:
ds_hard["train"][0]

In [ ]:
ds_toxic = load_dataset("bench-llm/or-bench", "or-bench-toxic")

In [ ]:
# Example
ds_toxic["train"][0]

## Load Gemma 3 Model
In order to be able to load a Gemma 3 model, terms of use needs to be accepted.

The setup is copied from https://huggingface.co/google/gemma-3-1b-it

In [ ]:

from transformers import pipeline

pipe = pipeline("text-generation", model="google/gemma-3-1b-it")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

# Model inference and Output classification
## Model inference

In [ ]:
# Run through the prompts


## Output Classification through LLM as a judge
The output classification prompt is copied from the official OR-bench github https://github.com/justincui03/or-bench/tree/main/rewritter response_checker/check_response.py [Linktext](https://)